# Supervised Fine-Tuning (SFT) with Serverless customization on SageMaker AI

## Lab 4a - Deploy to Amazon Bedrock via Custom Model Import

In this notebook, we deploy the fine-tuned LLM to Amazon Bedrock using [Custom Model Import](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-import-model.html). This provides fully serverless inference — no endpoints or infrastructure to manage.

**When to use this vs. Lab 4 (SageMaker Endpoint):**
- **Bedrock Custom Model Import (this notebook):** Fully serverless, no infrastructure to manage, billed per Custom Model Unit in 5-minute windows with scale-to-zero. Best when you want simplicity and don't need fine-grained control over the hosting environment.
- **SageMaker Endpoint (Lab 4):** Dedicated GPU instances, full control over instance type, scaling policies, and serving configuration. Best when you need predictable latency or cost optimization at high throughput.

***

### Prerequisites

#### Setup and dependencies

In [ ]:
%load_ext autoreload
%autoreload 2

import boto3
import os
from rich.pretty import pprint
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

#### Retrieve fine-tuned model artifacts

This retrieves the merged fine-tuned model artifacts registered in the Model Package Group from Lab 2. The merged model (in Hugging Face format) is what we'll import into Bedrock.

> **Note:** Edit `model_package_group_name` and `model_package_version` below if you used different values during fine-tuning.

In [ ]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup
from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

model_package_group_name = f"{base_model_id}-sft-mpg"
model_package_version = "1"

In [ ]:
from sagemaker.core import s3

model_package_group = ModelPackageGroup.get(model_package_group_name)

fine_tuned_model_package_group_arn = model_package_group.model_package_group_arn
print(f"Fine-tuned Model Package Group ARN: {fine_tuned_model_package_group_arn}")

fine_tuned_model_package_arn = f"{model_package_group.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
print(f"Fine-tuned Model Package ARN: {fine_tuned_model_package_arn}")

model_package = ModelPackage.get(fine_tuned_model_package_arn)

# get the merged model artifact and deploy it
merged_model_s3_uri = (
    s3.s3_path_join(
        model_package.inference_specification.containers[
            0
        ].model_data_source.s3_data_source.s3_uri,
        "checkpoints",
        "hf_merged",
    )
    + "/"
)
print(merged_model_s3_uri)

***

### Deploy as Custom Model Import on Amazon Bedrock

[Amazon Bedrock Custom Model Import](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-import-model.html) allows you to import fine-tuned models with custom weights into Bedrock for serverless inference. This eliminates the need to manage endpoints or infrastructure — Bedrock handles scaling automatically.

**Supported architectures include:** Llama, Mistral, Mixtral, Qwen (Qwen2, Qwen2.5, Qwen3), and others.

**Requirements:**
- Model files in Hugging Face format (`.safetensors`, `config.json`, `tokenizer.json`, etc.)
- Model stored in Amazon S3
- IAM role with Bedrock trust relationship and S3 read access

#### IAM role for Bedrock Custom Model Import

Bedrock requires an IAM role with a trust policy for `bedrock.amazonaws.com` and read access to the S3 bucket containing the model artifacts.

**Follow [Create a service role for model import](https://docs.aws.amazon.com/bedrock/latest/userguide/model-import-iam-role.html) to create this role**, then paste the role ARN below.

In [ ]:
import json
import time
from datetime import datetime
from urllib.parse import urlparse

bedrock_client = boto3.client("bedrock", region_name=sess.boto_region_name)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=sess.boto_region_name)

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
imported_model_name = f"{base_model_id}-sft-imported"
import_job_name = f"{base_model_id}-sft-import-job-{timestamp}"

# REQUIRED: Paste the ARN of the IAM role you created for Bedrock Custom Model Import
bedrock_import_role_arn = "<YOUR_BEDROCK_IMPORT_ROLE_ARN>"

assert bedrock_import_role_arn != "<YOUR_BEDROCK_IMPORT_ROLE_ARN>" and bedrock_import_role_arn.startswith("arn:aws:iam::"), (
    "You must provide a valid IAM role ARN for Bedrock Custom Model Import. "
    "Follow the instructions in the cell above to create one."
)

#### Submit Custom Model Import Job

This creates an import job that loads the fine-tuned model from S3 into Bedrock. The `merged_model_s3_uri` from earlier points to the merged Hugging Face model artifacts.

In [ ]:
print(f"Model S3 URI: {merged_model_s3_uri}")
print(f"Import job name: {import_job_name}")
print(f"Imported model name: {imported_model_name}")

create_job_response = bedrock_client.create_model_import_job(
    jobName=import_job_name,
    importedModelName=imported_model_name,
    roleArn=bedrock_import_role_arn,
    modelDataSource={"s3DataSource": {"s3Uri": merged_model_s3_uri}},
)

job_arn = create_job_response["jobArn"]
print(f"Import job ARN: {job_arn}")

#### Wait for import job to complete

The import job typically takes several minutes. We'll poll until it completes.

In [ ]:
while True:
    response = bedrock_client.get_model_import_job(jobIdentifier=job_arn)
    status = response["status"]
    print(f"Import job status: {status}")

    if status == "Completed":
        imported_model_arn = response["importedModelArn"]
        print(f"\nImport complete! Model ARN: {imported_model_arn}")
        break
    elif status == "Failed":
        print(f"\nImport failed: {response.get('failureMessage', 'Unknown error')}")
        break
    else:
        time.sleep(60)

#### Test the imported model

Invoke the imported model using `invoke_model` with the [OpenAI Chat Completion schema](https://docs.aws.amazon.com/bedrock/latest/userguide/invoke-imported-model.html). Bedrock automatically detects the request format based on the payload structure.

> **Note:** Qwen3 imported models do not support the Converse API ([docs](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-import-model.html)). Imported models may also return `ModelNotReadyException` if the model was removed for hardware optimization — the retry config below handles this automatically.

In [ ]:
prompt = """
Regarding the temporomandibular joint, which statements are true or false: 
Is the temporomandibular joint a synovial joint? 
Is the articular disc a remnant of the tendon of the medial pterygoid? 
Do gliding movements occur in the lower compartment and rotatory movements occur in the upper compartment? 
Is the joint capsule thick and tight in the lower part and loose and lax in the upper part? 
Does the sphenomandibular ligament act as a false support to the joint and attach to the angle of the mandible?
"""

In [ ]:
from botocore.config import Config

# Configure retries for ModelNotReadyException
retry_config = Config(retries={"total_max_attempts": 10, "mode": "standard"})
bedrock_runtime = boto3.client(
    "bedrock-runtime", region_name=sess.boto_region_name, config=retry_config
)

request_body = json.dumps({
    "messages": [
        {
            "role": "user",
            "content": prompt,
        }
    ],
    "max_tokens": 4096,
    "temperature": 0.3,
    "top_p": 0.9,
})

response = bedrock_runtime.invoke_model(
    modelId=imported_model_arn,
    body=request_body,
    accept="application/json",
    contentType="application/json",
)

response_body = json.loads(response["body"].read())
generated_text = response_body["choices"][0]["message"]["content"]
print(generated_text)

#### Delete Bedrock imported model

> **Important:** Bedrock Custom Model Import is billed per [Custom Model Unit (CMU)](https://docs.aws.amazon.com/bedrock/latest/userguide/import-model-calculate-cost.html) in 5-minute windows starting from the first invocation, plus monthly storage costs per CMU. Bedrock scales to zero after 5 minutes of inactivity, but **delete the imported model when you're done** to stop storage charges. See [Bedrock pricing](https://aws.amazon.com/bedrock/pricing/) for current rates.

In [ ]:
bedrock_client.delete_imported_model(modelIdentifier=imported_model_name)
print(f"Deleted imported model: {imported_model_name}")